# W02 라이브 · 검색식 조립기

슬라이드 18 과 **같은 규칙**(블록 안 OR · 블록 사이 AND · NOT 배제)을 `md_corpus_267.csv`(267편) 의 `topics` 열에 적용한다. 덱의 라이브는 비공개 코퍼스 색인 654편의 태그를 내장했고, 여기서는 공개된 267편 데이터로 같은 계산을 한다.

In [ ]:
import pandas as pd
df = pd.read_csv('data/md_corpus_267.csv')
df['tags'] = df['topics'].fillna('').map(lambda s: {x.strip() for x in s.split(';') if x.strip()})
all_tags = sorted({t for s in df['tags'] for t in s})
print(len(df), '편 · 태그', len(all_tags)); print(all_tags)

In [ ]:
def query(df, A=(), B=(), NOT=(), year_from=None):
    """(A1 OR A2 …) AND (B1 OR B2 …) NOT (N1 OR …) · 블록이 비어 있으면 그 조건은 통과."""
    def hit(tags, block): return (not block) or any(t in tags for t in block)
    m = df['tags'].map(lambda s: hit(s, A) and hit(s, B) and not any(t in s for t in NOT))
    if year_from: m &= df['year'] >= year_from
    return df[m]

# 초기 예시 — 덱 슬라이드 18 과 같은 구조 (태그 이름은 이 데이터의 것)
A = ['Moisture/Adhesion/Interface', 'Aggregate interaction']
B = ['Asphalt/Bitumen']
N = ['Machine Learning']
res = query(df, A, B, N)
print(f'( {" OR ".join(A)} ) AND ( {" OR ".join(B)} ) NOT ( {" OR ".join(N)} ) → {len(res)} 편 / {len(df)}')
print('A 만', len(query(df, A)), '· B 만', len(query(df, B=B)))

In [ ]:
# 해 보기: OR 를 늘리면(↑) · AND 블록을 더하면(↓) 편수가 어떻게 변하나
for A2 in ([A[0]], A, A + ['Aging/Oxidation']):
    print(len(A2), 'OR 항 →', len(query(df, A2, B, N)), '편')
print('AND 블록 추가(Cement/C-S-H) →', len(query(df, A, B + [], N)), '→', len(query(df, A, ['Cement/C-S-H'], N)), '편')

기록 3항목(검색식 · 실행 날짜 · 결과 건수)을 `out/query_log.md` 에 남겨 두면 다음 주에 같은 질의를 재실행할 수 있다.